# DDL: `dbspend360_total_sql_warehouse_spends`

Creates the final per-warehouse / per-day spend rollup that backs the SQL Warehouses tab.

Sibling of `dbspend360_total_pipeline_spends`, keyed on `(warehouse_id, usage_date)`. Unlike `pipeline_id`,
`warehouse_id` is account-unique (validated), so `workspace_id` is carried as a descriptive column only and is
not part of the key.

Warehouse metadata (`warehouse_name`, `warehouse_type`, `warehouse_size`, `creator_id`, `auto_stop_mins`,
`min_clusters`, `max_clusters`) is denormalized from `system.compute.warehouses` so the UI does not need a live
join.

Three-state snapshot handling:
- Active warehouse: `metadata_missing = FALSE`, `warehouse_deleted_at IS NULL`.
- Deleted but visible: `metadata_missing = FALSE`, `warehouse_deleted_at` populated (UI renders a
  "Deleted YYYY-MM-DD" badge).
- Metadata missing entirely: `metadata_missing = TRUE`, `warehouse_deleted_at IS NULL` (UI renders a
  "Metadata missing" badge and falls back to `"Warehouse {warehouse_id}"`).

**DBU-only: no cloud cost columns.** SQL warehouses run on Databricks-managed compute for all three types
(Classic, Pro, Serverless), so there is no customer-account VM spend to attribute. The DBU cost IS the complete
cost, hence `total_cost = databricks_cost` with no cloud component to add.

**Widgets**
- `catalog` - target Unity Catalog name
- `schema`  - target schema name within `catalog`

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")

In [ ]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

if not catalog or not schema:
    raise ValueError("Both `catalog` and `schema` widgets must be set.")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.dbspend360_total_sql_warehouse_spends (
  warehouse_id          STRING,
  usage_date            DATE,
  warehouse_name        STRING,
  warehouse_type        STRING,
  warehouse_size        STRING,
  creator_id            STRING,
  auto_stop_mins        INT,
  min_clusters          INT,
  max_clusters          INT,
  metadata_missing      BOOLEAN,
  warehouse_deleted_at  TIMESTAMP,
  databricks_cost       DOUBLE,
  total_cost            DOUBLE,
  currency              STRING,
  sku_name              STRING,
  workspace_id          STRING,
  workspace_covered     BOOLEAN,
  created_at            TIMESTAMP,
  updated_at            TIMESTAMP
)
CLUSTER BY AUTO

In [ ]:
dbutils.notebook.exit(f"{catalog}.{schema}.dbspend360_total_sql_warehouse_spends")